# Laboratorio 03 — Evaluador automático de prompts con métricas
### Python AI Developer 2026 · Capítulo 2: Prompt Engineering & Fine-tuning

**Tarea central:** extracción de datos estructurados desde texto libre en español.

Dado un correo de reclamo de un cliente bancario, el modelo debe extraer campos específicos en formato JSON válido. La métrica es determinista: o el JSON es correcto o no lo es — sin juez LLM, sin variabilidad.

**Objetivo real de este lab:** descubrir empíricamente qué técnica de prompting funciona mejor para *esta tarea específica* y entender por qué. No existe un ganador universal — esa es exactamente la lección.

Duración estimada: 90 min  
Setup: `uv add anthropic python-dotenv`

---
## Parte 1 — Setup

In [1]:
import os
import json
import time
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

print("Setup listo")

Setup listo


---
## Parte 2 — La tarea y el dataset de evaluación

Dado un correo de reclamo bancario en texto libre, extraer exactamente estos 5 campos:

```json
{
  "tipo_reclamo": "cobro_indebido | bloqueo_cuenta | error_transferencia | otro",
  "monto_afectado": <número o null si no se menciona>,
  "fecha_incidente": "YYYY-MM-DD o null si no se menciona",
  "canal_afectado": "app | web | cajero | ventanilla | otro",
  "urgencia": "alta | media | baja"
}
```

La métrica es **exactitud por campo**: porcentaje de campos extraídos correctamente sobre el total.

In [2]:
# Dataset de evaluación: 10 correos con extracción correcta definida por un humano
# Los correos tienen distintos niveles de ambigüedad y complejidad

correos_eval = [
    {
        "correo": "Buenos días, les escribo porque ayer 15 de enero me cobraron S/. 45.00 de comisión en mi app sin ninguna explicación. Necesito que me devuelvan ese dinero hoy mismo porque es urgente.",
        "correcto": {"tipo_reclamo": "cobro_indebido", "monto_afectado": 45.0, "fecha_incidente": "2024-01-15", "canal_afectado": "app", "urgencia": "alta"}
    },
    {
        "correo": "Mi cuenta fue bloqueada sin previo aviso desde el lunes pasado. No puedo acceder ni por la web ni por ningún lado. Por favor desbloquéenla cuando puedan.",
        "correcto": {"tipo_reclamo": "bloqueo_cuenta", "monto_afectado": None, "fecha_incidente": None, "canal_afectado": "web", "urgencia": "media"}
    },
    {
        "correo": "Hice una transferencia de 1200 soles el 3 de marzo por el cajero automático y el dinero salió de mi cuenta pero nunca llegó al destinatario. Llevo 5 días esperando respuesta.",
        "correcto": {"tipo_reclamo": "error_transferencia", "monto_afectado": 1200.0, "fecha_incidente": "2024-03-03", "canal_afectado": "cajero", "urgencia": "alta"}
    },
    {
        "correo": "Quiero felicitar al personal de ventanilla de la sucursal Miraflores, me atendieron muy bien el viernes pasado.",
        "correcto": {"tipo_reclamo": "otro", "monto_afectado": None, "fecha_incidente": None, "canal_afectado": "ventanilla", "urgencia": "baja"}
    },
    {
        "correo": "El 22/02 intenté pagar con mi tarjeta en la app y me aparece error 504. Ya son tres días así y tengo pagos pendientes de servicios.",
        "correcto": {"tipo_reclamo": "otro", "monto_afectado": None, "fecha_incidente": "2024-02-22", "canal_afectado": "app", "urgencia": "alta"}
    },
    {
        "correo": "Me llegó un estado de cuenta con un cargo de 350 dólares el 10 de enero que yo no reconozco. Por favor investiguen.",
        "correcto": {"tipo_reclamo": "cobro_indebido", "monto_afectado": 350.0, "fecha_incidente": "2024-01-10", "canal_afectado": "otro", "urgencia": "media"}
    },
    {
        "correo": "Desde ayer mi acceso a la banca web está bloqueado. No puedo ver mis saldos ni hacer ninguna operación. Es urgente porque tengo que pagar planillas mañana.",
        "correcto": {"tipo_reclamo": "bloqueo_cuenta", "monto_afectado": None, "fecha_incidente": None, "canal_afectado": "web", "urgencia": "alta"}
    },
    {
        "correo": "Buenos días. Quisiera saber si es posible obtener una constancia de no adeudo para trámites notariales. No hay apuro.",
        "correcto": {"tipo_reclamo": "otro", "monto_afectado": None, "fecha_incidente": None, "canal_afectado": "otro", "urgencia": "baja"}
    },
    {
        "correo": "Realicé una transferencia de S/. 890 el 5 de enero desde la app hacia mi cuenta del BCP pero el monto no llegó. Ya hablé con el banco receptor y dicen que no recibieron nada.",
        "correcto": {"tipo_reclamo": "error_transferencia", "monto_afectado": 890.0, "fecha_incidente": "2024-01-05", "canal_afectado": "app", "urgencia": "alta"}
    },
    {
        "correo": "En el cajero de San Isidro me cobró dos veces la misma operación de retiro de 200 soles el martes. Necesito que devuelvan el cobro duplicado a la brevedad.",
        "correcto": {"tipo_reclamo": "cobro_indebido", "monto_afectado": 200.0, "fecha_incidente": None, "canal_afectado": "cajero", "urgencia": "alta"}
    },
]

CAMPOS = ["tipo_reclamo", "monto_afectado", "fecha_incidente", "canal_afectado", "urgencia"]

print(f"Dataset: {len(correos_eval)} correos")
print(f"Campos a extraer: {CAMPOS}")
print(f"Total de extracciones a evaluar: {len(correos_eval) * len(CAMPOS)}")

Dataset: 10 correos
Campos a extraer: ['tipo_reclamo', 'monto_afectado', 'fecha_incidente', 'canal_afectado', 'urgencia']
Total de extracciones a evaluar: 50


---
## Parte 3 — Función de llamada y métrica de evaluación

La métrica es determinista: comparamos campo por campo con el valor correcto.
No hay juez LLM — o el campo está bien o no.

In [6]:
def llamar_modelo(system: str, user: str) -> dict:
    """Llama al modelo y devuelve el JSON extraído + tokens usados."""
    response = client.messages.create(
        model=MODELO,
        system=system,
        messages=[{"role": "user", "content": user}],
        temperature=0.1,  # baja temperatura para extracciones: queremos determinismo
        max_tokens=300,
    )
    texto = response.content[0].text.strip()
    texto = texto.replace("```json", "").replace("```", "").strip()

    try:
        datos = json.loads(texto)
    except json.JSONDecodeError:
        # si el modelo no devolvió JSON válido, registrar el fallo
        datos = {"_error": "JSON inválido", "_raw": texto}

    return {
        "datos": datos,
        "tokens_in": response.usage.input_tokens,
        "tokens_out": response.usage.output_tokens,
    }


def evaluar_extraccion(extraido: dict, correcto: dict) -> dict:
    """
    Compara campo por campo el JSON extraído con el correcto.
    Devuelve exactitud por campo y exactitud total.
    """
    if "_error" in extraido:
        # JSON inválido: todos los campos fallan
        return {"exactitud": 0.0, "campos_correctos": 0, "detalle": {c: False for c in CAMPOS}}

    detalle = {}
    for campo in CAMPOS:
        val_extraido = extraido.get(campo)
        val_correcto = correcto.get(campo)

        # normalizar para comparación: None == null, números como float
        if val_correcto is None:
            detalle[campo] = val_extraido is None
        elif isinstance(val_correcto, float):
            try:
                detalle[campo] = abs(float(val_extraido) - val_correcto) < 0.01
            except (TypeError, ValueError):
                detalle[campo] = False
        else:
            # comparar strings en minúsculas para evitar fallos por capitalización
            detalle[campo] = str(val_extraido).lower().strip() == str(val_correcto).lower().strip()

    correctos = sum(detalle.values())
    return {
        "exactitud": correctos / len(CAMPOS),
        "campos_correctos": correctos,
        "detalle": detalle,
    }


def evaluar_prompt_completo(nombre: str, system: str, user_template: str) -> dict:
    """Evalúa un prompt en todos los correos y agrega las métricas."""
    exactitudes = []
    tokens_in_total = 0
    json_invalidos = 0

    for correo in correos_eval:
        resultado = llamar_modelo(system, user_template.replace("{correo}", correo["correo"]))
        ev = evaluar_extraccion(resultado["datos"], correo["correcto"])
        exactitudes.append(ev["exactitud"])
        tokens_in_total += resultado["tokens_in"]
        if "_error" in resultado["datos"]:
            json_invalidos += 1
        time.sleep(0.3)

    exactitud_media = sum(exactitudes) / len(exactitudes)
    return {
        "nombre": nombre,
        "exactitud_media": exactitud_media,
        "exactitudes": exactitudes,
        "tokens_in_promedio": tokens_in_total / len(correos_eval),
        "json_invalidos": json_invalidos,
    }


print("Funciones definidas")

Funciones definidas


---
## Parte 4 — Las 5 variantes de prompt

Cada variante agrega una capa de instrucción sobre la anterior.
Veremos cómo cada capa impacta la exactitud de extracción.

In [4]:
# El schema JSON que queremos extraer — lo definimos una vez y lo reutilizamos
SCHEMA = """{
  "tipo_reclamo": "cobro_indebido | bloqueo_cuenta | error_transferencia | otro",
  "monto_afectado": <número o null>,
  "fecha_incidente": "YYYY-MM-DD o null",
  "canal_afectado": "app | web | cajero | ventanilla | otro",
  "urgencia": "alta | media | baja"
}"""

prompts = [
    {
        "nombre": "01_zero_shot",
        "descripcion": "Zero-shot — solo el schema, sin instrucciones adicionales",
        "system": "Eres un asistente útil. Responde siempre en JSON.",
        "user": f"Extrae los datos de este correo bancario en JSON con este schema:\n{SCHEMA}\n\nCorreo: {{correo}}",
    },
    {
        "nombre": "02_rol",
        "descripcion": "Con rol técnico de analista bancario",
        "system": (
            "Eres un analista de reclamos bancarios con 10 años de experiencia. "
            "Tu tarea es extraer información estructurada de correos de clientes con máxima precisión. "
            "Responde ÚNICAMENTE con JSON válido, sin texto adicional."
        ),
        "user": f"Extrae los datos del siguiente correo usando exactamente este schema:\n{SCHEMA}\n\nCorreo: {{correo}}",
    },
    {
        "nombre": "03_few_shot",
        "descripcion": "Few-shot con un ejemplo de extracción correcta",
        "system": "Eres un asistente de análisis bancario. Responde solo con JSON válido.",
        "user": (
            f"Extrae datos de correos bancarios con este schema:\n{SCHEMA}\n\n"
            "Ejemplo:\n"
            'Correo: "Me cobraron 80 soles de más en la app el 5 de enero, es urgente."\n'
            '{"tipo_reclamo": "cobro_indebido", "monto_afectado": 80.0, '
            '"fecha_incidente": "2024-01-05", "canal_afectado": "app", "urgencia": "alta"}\n\n'
            "Ahora extrae este correo:\nCorreo: {correo}"
        ),
    },
    {
        "nombre": "04_cot",
        "descripcion": "Chain-of-Thought — razonar antes de extraer",
        "system": (
            "Eres un analista de reclamos bancarios. "
            "Responde SOLO con JSON válido. Nada más."
        ),
        "user": (
            f"Extrae los datos del correo con este schema:\n{SCHEMA}\n\n"
            "Antes de responder, identifica mentalmente:\n"
            "1. ¿Qué tipo de problema describe el cliente?\n"
            "2. ¿Menciona algún monto? ¿En qué moneda?\n"
            "3. ¿Hay una fecha específica o solo referencia relativa?\n"
            "4. ¿Qué canal usó el cliente?\n"
            "5. ¿El lenguaje indica urgencia?\n"
            "Luego responde SOLO con el JSON, sin incluir el razonamiento.\n\n"
            "Correo: {correo}"
        ),
    },
    {
        "nombre": "05_combinado",
        "descripcion": "Combinado: rol + schema estricto + CoT + restricciones explícitas",
        "system": (
            "Eres un analista senior de reclamos bancarios. "
            "Extraes información con precisión quirúrgica. "
            "Reglas absolutas:\n"
            "- Responde ÚNICAMENTE con JSON válido parseable por json.loads()\n"
            "- NUNCA inventes datos que no estén explícitamente en el texto\n"
            "- Si un campo no se menciona, usa null exactamente\n"
            "- Los montos van como número (sin símbolo de moneda)\n"
            "- Las fechas van en formato YYYY-MM-DD o null\n"
            "- No agregues campos extra al schema"
        ),
        "user": (
            f"Schema requerido (usar EXACTAMENTE estos campos y valores):\n{SCHEMA}\n\n"
            "Proceso de extracción:\n"
            "1. Lee el correo y determina el tipo de reclamo\n"
            "2. Busca montos numéricos explícitos (ignorar moneda)\n"
            "3. Busca fechas en cualquier formato y conviértelas a YYYY-MM-DD\n"
            "4. Identifica el canal mencionado\n"
            "5. Evalúa la urgencia por el lenguaje (palabras como 'urgente', 'hoy', 'mañana' → alta)\n"
            "Devuelve SOLO el JSON.\n\n"
            "Correo: {correo}"
        ),
    },
]

print(f"Definidos {len(prompts)} prompts")
for p in prompts:
    print(f"  {p['nombre']}: {p['descripcion']}")

Definidos 5 prompts
  01_zero_shot: Zero-shot — solo el schema, sin instrucciones adicionales
  02_rol: Con rol técnico de analista bancario
  03_few_shot: Few-shot con un ejemplo de extracción correcta
  04_cot: Chain-of-Thought — razonar antes de extraer
  05_combinado: Combinado: rol + schema estricto + CoT + restricciones explícitas


---
## Parte 5 — Evaluación de los 5 prompts

In [12]:
resultados = []

for p in prompts:
    print(f"Evaluando {p['nombre']}...")
    r = evaluar_prompt_completo(p["nombre"], p["system"], p["user"])
    resultados.append(r)
    print(f"  Exactitud: {r['exactitud_media']*100:.1f}%  "
          f"tokens_in promedio: {r['tokens_in_promedio']:.0f}  "
          f"JSON inválidos: {r['json_invalidos']}")

print()
mejor = max(resultados, key=lambda x: x["exactitud_media"])
print(f"Mejor prompt: {mejor['nombre']} — {mejor['exactitud_media']*100:.1f}%")
peor = min(resultados, key=lambda x: x["exactitud_media"])
print(f"Mejor prompt: {peor['nombre']} — {peor['exactitud_media']*100:.1f}%")

Evaluando 01_zero_shot...
  Exactitud: 76.0%  tokens_in promedio: 198  JSON inválidos: 2
Evaluando 02_rol...
  Exactitud: 96.0%  tokens_in promedio: 237  JSON inválidos: 0
Evaluando 03_few_shot...
  Exactitud: 98.0%  tokens_in promedio: 302  JSON inválidos: 0
Evaluando 04_cot...
  Exactitud: 98.0%  tokens_in promedio: 321  JSON inválidos: 0
Evaluando 05_combinado...
  Exactitud: 94.0%  tokens_in promedio: 429  JSON inválidos: 0

Mejor prompt: 03_few_shot — 98.0%
Mejor prompt: 01_zero_shot — 76.0%


---
## Parte 6 — Tabla comparativa y análisis por campo

In [8]:
import pandas as pd

filas = []
for r in resultados:
    costo = r["tokens_in_promedio"] * 0.80 / 1_000_000  # Claude Haiku input price
    filas.append({
        "Prompt": r["nombre"],
        "Exactitud %": round(r["exactitud_media"] * 100, 1),
        "Tokens input": round(r["tokens_in_promedio"]),
        "JSON inválidos": r["json_invalidos"],
        "Costo/1000 correos $": round(costo * 1000, 4),
    })

df = pd.DataFrame(filas)
print(df.to_string(index=False))

      Prompt  Exactitud %  Tokens input  JSON inválidos  Costo/1000 correos $
01_zero_shot         76.0           198               2                0.1586
      02_rol         96.0           237               0                0.1898
 03_few_shot         98.0           302               0                0.2418
      04_cot         98.0           321               0                0.2570
05_combinado         94.0           429               0                0.3434


In [25]:
# Analizar el mejor y el peor prompt campo por campo
# para entender QUÉ está fallando, no solo cuánto

def exactitud_por_campo(nombre_prompt: str, system: str, user_template: str) -> dict:
    """Calcula la exactitud de cada campo por separado."""
    aciertos_por_campo = {c: 0 for c in CAMPOS}

    for correo in correos_eval:
        resultado = llamar_modelo(system, user_template.replace("{correo}", correo["correo"]))
        ev = evaluar_extraccion(resultado["datos"], correo["correcto"])
        for campo in CAMPOS:
            if ev["detalle"][campo]:
                aciertos_por_campo[campo] += 1
        time.sleep(0.3)

    return {c: aciertos_por_campo[c] / len(correos_eval) for c in CAMPOS}


print(f"Análisis por campo — prompt 01 {mejor['nombre']} vs {peor['nombre']}:")
print()

peor = list(filter(lambda x: x["nombre"] == peor['nombre'],prompts))[0]
mejor = list(filter(lambda x: x["nombre"] == mejor['nombre'],prompts))[0]

campos_zero = exactitud_por_campo(peor["nombre"], peor["system"], peor["user"])
campos_comb = exactitud_por_campo(mejor["nombre"], mejor["system"], mejor["user"])

print(f"{'Campo':<25} {peor['nombre']:>12} {mejor['nombre']:>12} {'Diferencia':>12}")
print("-" * 62)
for campo in CAMPOS:
    diff = campos_comb[campo] - campos_zero[campo]
    print(f"  {campo:<23} {campos_zero[campo]*100:>10.0f}%  {campos_comb[campo]*100:>10.0f}%  {diff*100:>+10.0f}%")

Análisis por campo — prompt 01 03_few_shot vs 01_zero_shot:

Campo                     01_zero_shot  03_few_shot   Diferencia
--------------------------------------------------------------
  tipo_reclamo                    70%          90%         +20%
  monto_afectado                  80%         100%         +20%
  fecha_incidente                 80%         100%         +20%
  canal_afectado                  80%         100%         +20%
  urgencia                        70%         100%         +30%


---
## Parte 7 — ¿Qué aprendemos de los resultados?

Esta celda conecta los resultados con los conceptos clave de prompt engineering.

**El hallazgo más importante de este lab no es cuál prompt ganó — es cuánto ganó.**

Si la diferencia entre el zero-shot y el combinado es pequeña (menos de 10 puntos porcentuales), eso revela algo fundamental: en tareas bien definidas con un schema claro, un modelo suficientemente capaz ya extrae la información correctamente con instrucciones mínimas. Las técnicas adicionales (CoT, few-shot, restricciones) ayudan en los casos borde, no en el caso general.

Esto tiene implicaciones directas para producción:
- Un prompt más complejo cuesta más tokens de input en cada llamada
- Si la ganancia de calidad es marginal, el costo adicional puede no justificarse
- El punto de inflexión es diferente para cada tarea y cada modelo

La conclusión práctica: **empezar siempre con el prompt más simple posible, medir, y agregar complejidad solo donde los datos muestran que es necesario.**

In [28]:
# Probar un caso difícil manualmente con todos los prompts
# para ver exactamente cómo difieren las respuestas

CORREO_DIFICIL = (
    "Estimados señores, el martes de la semana pasada intenté sacar plata "
    "del cajero de la esquina y me descontó doscientos soles pero no salió nada. "
    "Por favor ayúdenme."
)
# Nota: 'doscientos' no es un número, 'martes de la semana pasada' no es fecha exacta

print(f"Correo difícil: '{CORREO_DIFICIL[:80]}...'")
print()

for p in [prompts[0], prompts[2], prompts[4]]:
    resultado = llamar_modelo(p["system"], p["user"].replace("{correo}", CORREO_DIFICIL))
    print(f"--- {p['nombre']} ---")
    print(json.dumps(resultado["datos"], ensure_ascii=False, indent=2))
    print()
    time.sleep(0.5)

print("Observar: el combinado maneja mejor los montos en texto y las fechas relativas.")
print("El zero-shot tiende a inventar valores o a devolver formatos incorrectos.")

Correo difícil: 'Estimados señores, el martes de la semana pasada intenté sacar plata del cajero ...'

--- 01_zero_shot ---
{
  "_error": "JSON inválido",
  "_raw": "{\n  \"tipo_reclamo\": \"cobro_indebido\",\n  \"monto_afectado\": 200,\n  \"fecha_incidente\": null,\n  \"canal_afectado\": \"cajero\",\n  \"urgencia\": \"alta\"\n}\n\n\n**Notas:**\n- `tipo_reclamo`: Se clasificó como \"cobro_indebido\" porque se descontó dinero sin entregar el servicio\n- `monto_afectado`: 200 soles (doscientos soles mencionados)\n- `fecha_incidente`: null (solo dice \"martes de la semana pasada\" sin fecha específica)\n- `canal_afectado`: \"cajero\" (cajero automático)\n- `urgencia`: \"alta\" (pérdida de dinero sin recibir el servicio)"
}

--- 03_few_shot ---
{
  "tipo_reclamo": "error_transferencia",
  "monto_afectado": 200.0,
  "fecha_incidente": null,
  "canal_afectado": "cajero",
  "urgencia": "alta"
}

--- 05_combinado ---
{
  "tipo_reclamo": "cobro_indebido",
  "monto_afectado": 200,
  "fecha_incid

In [29]:
# Resumen del tradeoff costo vs calidad
print("TRADEOFF: calidad vs costo de tokens")
print("=" * 55)
print()
for r in resultados:
    costo_1000 = r["tokens_in_promedio"] * 0.80 / 1_000_000 * 1000
    print(f"  {r['nombre']:20s}  exactitud={r['exactitud_media']*100:.1f}%  "
          f"tokens={r['tokens_in_promedio']:.0f}  costo/1k=${costo_1000:.4f}")

print()
print("El prompt combinado usa más tokens de input pero produce JSON más correcto.")
print("En producción con 100k correos/mes, la diferencia de costo es mínima")
print("comparada con el costo de errores de extracción no detectados.")

TRADEOFF: calidad vs costo de tokens

  01_zero_shot          exactitud=76.0%  tokens=198  costo/1k=$0.1586
  02_rol                exactitud=96.0%  tokens=237  costo/1k=$0.1898
  03_few_shot           exactitud=98.0%  tokens=302  costo/1k=$0.2418
  04_cot                exactitud=98.0%  tokens=321  costo/1k=$0.2570
  05_combinado          exactitud=94.0%  tokens=429  costo/1k=$0.3434

El prompt combinado usa más tokens de input pero produce JSON más correcto.
En producción con 100k correos/mes, la diferencia de costo es mínima
comparada con el costo de errores de extracción no detectados.


---
## Reflexión del alumno

1. ¿Cuál prompt obtuvo mejor exactitud en tu ejecución? ¿La diferencia con el segundo mejor fue grande o pequeña? ¿A qué lo atribuyes?

*(escribe aquí)*

---

2. Observa la columna de tokens de input en la tabla comparativa. ¿El prompt que más tokens consume es necesariamente el mejor? ¿Qué conclusión sacas sobre el tradeoff costo/calidad?

*(escribe aquí)*

---

3. Si tuvieras que elegir un prompt para procesar 500,000 correos al mes en producción, ¿cuál elegirías y por qué? Considera exactitud, costo de tokens y mantenibilidad del prompt.

*(escribe aquí)*

---

4. ¿En qué tipo de tarea crees que el prompt combinado sí mostraría una ventaja clara y consistente sobre el zero-shot? Da un ejemplo concreto.

*(escribe aquí)*

---
*Laboratorio 02 — Python AI Developer 2026 · IES Cibertec*